In [4]:
import polars as pl
import glob
from pathlib import Path
pl.Config.set_tbl_cols(200).set_fmt_str_lengths(150).set_tbl_rows(100)

polars.config.Config

In [7]:
file_path = "../data/raw_data/visados_construccion"

def processar_arxiu(directory_file):
    try:
        df_raw = pl.read_excel(
            directory_file,
            has_header=False,
            engine="calamine",
            infer_schema_length=0 
        )
        
        if df_raw.height < 10: return None
        try:
            source = str(df_raw.item(-1, 0)).strip()
            place = str(df_raw.item(2, 1)).strip()
        except:
            source, place = "DESCONOCIDO", "DESCONOCIDO"
        df_clean = (
            df_raw
            .slice(8)
            .select([
                pl.nth(0).alias("year_region"),
                pl.nth(2).alias("mes"),
                pl.nth(3).alias("n_viv_unifam"),    
                pl.nth(4).alias("n_viv_bloque"),  
                pl.nth(5).alias("n_viv_otros_edificios"),    
                pl.nth(6).alias("unifam_superficie"),   
                pl.nth(7).alias("bloque_superficie"),    
                pl.nth(10).alias("n_viv_ampliacion_reforma"),   
                pl.nth(11).alias("total")    
            ])
            .with_columns(
                pl.col("year_region").forward_fill(),
                pl.lit(source).alias("fuente"),
                pl.lit(place).alias("place")
            )
            .filter(
                (pl.col("year_region") != source) &
                (pl.col("year_region").is_not_null())
            )
            .with_columns(
                pl.col("year_region").cast(pl.Int32, strict=False)
            )
            .filter(pl.col("year_region").is_not_null())
        )

        return df_clean

    except Exception as e:
        print(f"Error a {directory_file.name}: {e}")
        return None

ruta = Path(file_path)
files = list(ruta.rglob("*.XLS")) + list(ruta.rglob("*.xls"))
llista_dfs = []

for file in files:
    df = processar_arxiu(file)
    if df is not None:
        llista_dfs.append(df)

if llista_dfs:
    df_total = pl.concat(llista_dfs, how="vertical_relaxed")
    print(f"\n📊 TOTAL: {df_total.height} files.")
    df_total.write_csv("../data/clean_data/visados_construccion.csv")

Invalid ifmt value: '0'
Invalid ifmt value: '1'
Invalid ifmt value: '2'
Invalid ifmt value: '3'
Invalid ifmt value: '4'
Invalid ifmt value: '9'
Invalid ifmt value: '11'
Invalid ifmt value: '12'
Invalid ifmt value: '13'
Invalid ifmt value: '14'
Invalid ifmt value: '15'
Invalid ifmt value: '16'
Invalid ifmt value: '17'
Invalid ifmt value: '18'
Invalid ifmt value: '19'
Invalid ifmt value: '20'
Invalid ifmt value: '21'
Invalid ifmt value: '22'
Invalid ifmt value: '37'
Invalid ifmt value: '38'
Invalid ifmt value: '39'
Invalid ifmt value: '40'
Invalid ifmt value: '45'
Invalid ifmt value: '46'
Invalid ifmt value: '47'
Invalid ifmt value: '48'
Invalid ifmt value: '49'
Invalid ifmt value: '0'
Invalid ifmt value: '1'
Invalid ifmt value: '2'
Invalid ifmt value: '3'
Invalid ifmt value: '4'
Invalid ifmt value: '9'
Invalid ifmt value: '11'
Invalid ifmt value: '12'
Invalid ifmt value: '13'
Invalid ifmt value: '14'
Invalid ifmt value: '15'
Invalid ifmt value: '16'
Invalid ifmt value: '17'
Invalid ifmt


📊 TOTAL: 20450 files.


In [6]:
df_total.sample(10)

year_region,mes,n_viv_unifam,n_viv_bloque,n_viv_otros_edificios,unifam_superficie,bloque_superficie,n_viv_ampliacion_reforma,total,fuente,place
i32,str,str,str,str,str,str,str,str,str,str
2017,"""Jun""","""10""","""68""","""0""","""208.5""","""106.808823""","""24""","""102""","""Fuente: M. Transportes y Movilidad Sostenible Fecha: 27/01/2026""","""Castellón"""
2017,"""Jul""","""47""","""109""","""0""","""197.127659""","""133.733944""","""45""","""201""","""Fuente: M. Transportes y Movilidad Sostenible Fecha: 27/01/2026""","""A Coruña"""
2011,"""Jul""","""34""","""4""","""0""","""183.029411""","""77.5""","""12""","""50""","""Fuente: M. Transportes y Movilidad Sostenible Fecha: 27/01/2026""","""León"""
2014,"""Nov""","""10""","""0""","""0""","""122.4""","""0""","""6""","""16""","""Fuente: M. Transportes y Movilidad Sostenible Fecha: 27/01/2026""","""Ávila"""
2001,"""Oct""","""525""","""481""","""4""","""134.565714""","""95.68607""","""49""","""1059""","""Fuente: M. Transportes y Movilidad Sostenible Fecha: 27/01/2026""","""Granada"""
1997,"""Jul""","""286""","""99""","""0""","""140.618881""","""113.272727""","""62""","""447""","""Fuente: M. Transportes y Movilidad Sostenible Fecha: 27/01/2026""","""Tarragona"""
2022,"""Jun""","""33""","""228""","""0""","""232.696969""","""116.241228""","""40""","""301""","""Fuente: M. Transportes y Movilidad Sostenible Fecha: 27/01/2026""","""Córdoba"""
2024,"""Ene""","""20""","""0""","""0""","""186.6""","""0""","""21""","""41""","""Fuente: M. Transportes y Movilidad Sostenible Fecha: 27/01/2026""","""Salamanca"""
1999,"""May""","""435""","""878""","""0""","""130.848275""","""94.648063""","""25""","""1338""","""Fuente: M. Transportes y Movilidad Sostenible Fecha: 27/01/2026""","""Cádiz"""
